# Lagrangian Tracker Validation: Synthetic Alfvén Wave

End-to-end validation of the Lagrangian trajectory tracer using synthetic simulation data
with a known shear Alfvén wave.

**Setup:**
- Uniform background: **B** = B0 ŷ (magnetic field along +y), **E** = −E0 ẑ (electric field along −z)
- E×B drift: v_x = E0/B0 in +x, v_y = 0
- Superimposed shear Alfvén wave propagating along +y (along B0)

**Key expectation:** The E×B drift is along x, but the wave propagates along y.
Since the drift has no component along the wave propagation direction,
the Lagrangian (co-moving) frequency should equal the Eulerian (lab-frame) frequency.
Both PSDs of Bx should show a peak at f = ω/(2π).

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from reconn_wave_power.spectrum import compute_psd_time
from reconn_wave_power.lagrangian import (
    compute_exb_velocity,
    trace_trajectory,
    sample_along_trajectory,
    lagrangian_psd,
)

%matplotlib inline

## Wave & Grid Parameters

In [ ]:
# Background fields
B0 = 1.0          # Background magnetic field magnitude (along +y)
E0 = 0.1          # Background electric field magnitude (along -z)
V_A = 1.0         # Alfven speed

# Wave parameters
LX = 32.0         # Domain size in x
LY = 32.0         # Domain size in y
WAVELENGTH = LY / 2  # 2 wavelengths fit in the y-domain
K_Y = 2 * np.pi / WAVELENGTH
OMEGA = K_Y * V_A  # Dispersion relation: omega = k_y * v_A
F0 = OMEGA / (2 * np.pi)  # Expected frequency in cycles/unit time
DB = B0 / 10       # Wave amplitude (small perturbation)

# Grid parameters
NX = 64
NY = 64
DX = LX / NX       # = 0.5
DY = LY / NY       # = 0.5

# Time parameters
DT_SIM = 0.05      # Internal simulation timestep
DT_OUTPUT = 0.1    # Output cadence (every 2nd internal step)
NT = 480           # Number of output frames (3 full wave periods)

# PSD segment length chosen so f0 lands exactly on a frequency bin:
# df = 1 / (NPERSEG * DT_OUTPUT) = 1/16 = 0.0625 = f0
NPERSEG = 160

print(f"Domain: {LX} x {LY}, grid: {NX} x {NY}, dx = {DX}, dy = {DY}")
print(f"Output cadence: {DT_OUTPUT} (sim dt = {DT_SIM}, differs from output cadence)")
print(f"Time range: 0 to {NT * DT_OUTPUT:.1f} ({NT} frames)")
print(f"")
print(f"B0 = {B0}, E0 = {E0}, drift speed = E0/B0 = {E0/B0}")
print(f"Wavelength = {WAVELENGTH}, k_y = {K_Y:.4f} rad/unit")
print(f"omega = {OMEGA:.4f} rad/unit time, f0 = {F0:.4f} cycles/unit time")
print(f"Wave period T = {WAVELENGTH/V_A:.1f}, {NT*DT_OUTPUT/(WAVELENGTH/V_A):.0f} full periods in dataset")
print(f"dB/B0 = {DB/B0}")
print(f"PSD nperseg = {NPERSEG}, df = {1/(NPERSEG*DT_OUTPUT):.4f}")

## Build Synthetic Dataset

In [ ]:
x = np.arange(NX) * DX
y = np.arange(NY) * DY
t = np.arange(NT) * DT_OUTPUT

# Broadcast y and t for wave computation: shape (nt, 1, ny)
Y = y[np.newaxis, np.newaxis, :]   # (1, 1, ny)
T = t[:, np.newaxis, np.newaxis]   # (nt, 1, 1)
phase = K_Y * Y - OMEGA * T        # (nt, 1, ny), broadcasts over x

# Build all 6 field components with shape (nt, nx, ny)
wave = DB * np.sin(phase)           # (nt, 1, ny) -> broadcasts to (nt, nx, ny)
zeros = np.zeros((NT, NX, NY))

ds = xr.Dataset(
    {
        "Bx": (("time", "x", "y"), np.broadcast_to(wave, (NT, NX, NY)).copy()),
        "By": (("time", "x", "y"), np.full((NT, NX, NY), B0)),
        "Bz": (("time", "x", "y"), zeros.copy()),
        "Ex": (("time", "x", "y"), zeros.copy()),
        "Ey": (("time", "x", "y"), zeros.copy()),
        "Ez": (("time", "x", "y"), np.broadcast_to(-E0 + V_A * wave, (NT, NX, NY)).copy()),
    },
    coords={"time": t, "x": x, "y": y},
)

print(ds)

## 1. Field Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

t_snap = 0

# Bx snapshot
ax = axes[0]
bx_snap = ds["Bx"].isel(time=t_snap).values.T
im = ax.pcolormesh(x, y, bx_snap, cmap="RdBu_r", shading="auto")
plt.colorbar(im, ax=ax, label="Bx")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Bx(x, y) at t = {t[t_snap]:.1f}")
ax.set_aspect("equal")

# Ez snapshot
ax = axes[1]
ez_snap = ds["Ez"].isel(time=t_snap).values.T
im = ax.pcolormesh(x, y, ez_snap, cmap="RdBu_r", shading="auto")
plt.colorbar(im, ax=ax, label="Ez")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Ez(x, y) at t = {t[t_snap]:.1f}")
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

## 2. E×B Drift Velocity

In [ ]:
vx, vy = compute_exb_velocity(ds, time_idx=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
im = ax.pcolormesh(x, y, vx.values.T, cmap="RdBu_r", shading="auto")
plt.colorbar(im, ax=ax, label="vx")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("E×B drift: vx")
ax.set_aspect("equal")

ax = axes[1]
im = ax.pcolormesh(x, y, vy.values.T, cmap="RdBu_r", shading="auto")
plt.colorbar(im, ax=ax, label="vy")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("E×B drift: vy")
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

print(f"Mean vx = {float(vx.mean()):.6f}  (expected E0/B0 = {E0/B0:.4f})")
print(f"Mean vy = {float(vy.mean()):.6f}  (expected ~ 0)")

## 3. Trajectory Tracing

In [ ]:
# Trace trajectories from several starting positions
starts = [
    (4.0, 4.0),
    (4.0, 12.0),
    (16.0, 8.0),
    (28.0, 24.0),
]

trajectories = []
for x0, y0 in starts:
    xt, yt, tt = trace_trajectory(ds, x0, y0, t0_idx=0)
    trajectories.append((xt, yt, tt))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# x(t)
ax = axes[0]
for i, (xt, yt, tt) in enumerate(trajectories):
    ax.plot(tt, xt, label=f"start ({starts[i][0]}, {starts[i][1]})")
ax.set_xlabel("Time")
ax.set_ylabel("x position")
ax.set_title("x(t) — should show linear drift with small wiggles")
ax.legend(fontsize=8)

# y(t)
ax = axes[1]
for i, (xt, yt, tt) in enumerate(trajectories):
    ax.plot(tt, yt, label=f"start ({starts[i][0]}, {starts[i][1]})")
ax.set_xlabel("Time")
ax.set_ylabel("y position")
ax.set_title("y(t) — small wave-induced oscillations")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4. Field Sampling Along Trajectory

In [ ]:
# Sample Bx along the first trajectory
xt, yt, tt = trajectories[0]
bx_sampled = sample_along_trajectory(ds, "Bx", xt, yt, tt)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(bx_sampled.coords["time"].values, bx_sampled.values, lw=0.8)
ax.set_xlabel("Time")
ax.set_ylabel("Bx")
ax.set_title("Bx sampled along Lagrangian trajectory")
ax.axhline(0, color="gray", lw=0.5, ls="--")
plt.tight_layout()
plt.show()

print(f"Expected wave period: {WAVELENGTH/V_A:.1f}")
print(f"Expected amplitude: +/- {DB:.2f}")

## 5. Lagrangian vs Eulerian PSD

In [ ]:
# --- Eulerian PSD: Bx time series at a fixed point ---
ix_fixed = NX // 4
iy_fixed = NY // 4
bx_euler = ds["Bx"].isel(x=ix_fixed, y=iy_fixed)
f_euler, pxx_euler = compute_psd_time(bx_euler, dt=DT_OUTPUT, method="welch", nperseg=NPERSEG)

# --- Lagrangian PSD ---
x0_lag, y0_lag = 4.0, 4.0
f_lag, pxx_lag, traj_info = lagrangian_psd(
    ds, "Bx", x0_lag, y0_lag, t0_idx=0, dt=DT_OUTPUT, method="fft", nperseg=NPERSEG,
)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Eulerian
ax = axes[0]
ax.semilogy(f_euler, pxx_euler, "b-", lw=1)
ax.axvline(F0, color="r", ls="--", lw=2, label=f"Expected f0 = {F0:.4f}")
ax.set_xlabel("Frequency [cycles/unit time]")
ax.set_ylabel("PSD")
ax.set_title("Eulerian PSD (fixed point)")
ax.legend()
ax.set_xlim(0, 0.5)

# Lagrangian
ax = axes[1]
ax.semilogy(f_lag, pxx_lag, "g-", lw=1)
ax.axvline(F0, color="r", ls="--", lw=2, label=f"Expected f0 = {F0:.4f}")
ax.set_xlabel("Frequency [cycles/unit time]")
ax.set_ylabel("PSD")
ax.set_title("Lagrangian PSD (co-moving frame)")
ax.legend()
ax.set_xlim(0, 0.5)

plt.tight_layout()
plt.show()

# Verify peaks
euler_peak_f = f_euler[np.argmax(pxx_euler)]
lag_peak_f = f_lag[np.argmax(pxx_lag)]
print(f"Expected frequency: f0 = {F0:.4f} cycles/unit time")
print(f"Eulerian peak:  f = {euler_peak_f:.4f}")
print(f"Lagrangian peak: f = {lag_peak_f:.4f}")
print(f"Match: Eulerian and Lagrangian frequencies agree "
      f"(drift perpendicular to wave propagation)")

## Summary

This notebook validates the Lagrangian trajectory tracer against a known analytic solution:

1. **E×B drift is correct** — mean vx matches E0/B0, vy is near zero
2. **Trajectories follow expected drift** — linear x-drift with small wave-induced perturbations
3. **Bx along trajectory shows wave oscillation** — clear sinusoidal signal at the Alfvén wave frequency
4. **Lagrangian PSD recovers the correct frequency** — peak at f0 = ω/(2π)
5. **Lagrangian and Eulerian frequencies match** — as expected, since the E×B drift is perpendicular to the wave propagation direction

In [ ]:
lagrangian_psd??

In [ ]:
vx, vy = compute_exb_velocity(ds, time_idx=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
im = ax.pcolormesh(x, y, vx.values.T, cmap="RdBu_r", shading="auto")
plt.colorbar(im, ax=ax, label="vx")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("E×B drift: vx")
ax.set_aspect("equal")

ax = axes[1]
im = ax.pcolormesh(x, y, vy.values.T, cmap="RdBu_r", shading="auto")
plt.colorbar(im, ax=ax, label="vy")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("E×B drift: vy")
ax.set_aspect("equal")

for i, (xt, yt, tt) in enumerate(trajectories):
    ax.plot(xt, yt)

plt.tight_layout()
plt.show()

print(f"Mean vx = {float(vx.mean()):.6f}  (expected E0/B0 = {E0/B0:.4f})")
print(f"Mean vy = {float(vy.mean()):.6f}  (expected ~ 0)")